# DEMO 6: Working with Complex Data Types in SQL

In a traditional database, handling nested or semi-structured data (like JSON) is painful — you’d typically resort to string parsing or regex. Databricks SQL makes this intuitive.

In this demo, you’ll learn:
- **Dot notation** — querying nested STRUCT columns like `address.city`
- **Colon syntax** — extracting fields from raw JSON strings
- **`from_json()`** — converting a JSON string into a proper typed structure
- **`to_json()`** — collapsing a structure back into a JSON string
- **Arrays and `EXPLODE`** — working with list-type columns

---

In [0]:
%run ./demo_6_setup

## Our Data

The setup created two tables:

- **`customer_profiles`** — has properly typed STRUCT columns (`address`, `preferences`) and an ARRAY column (`tags`)
- **`raw_events`** — has a `payload` column containing raw JSON strings (different structure per event type)

Let’s start by looking at the raw data to understand what we’re working with.

---

In [0]:
%sql
-- Let's see what customer_profiles looks like
-- Notice the address and preferences columns contain nested data
SELECT * FROM customer_profiles;

In [0]:
%sql
-- And raw_events has a payload column that's just a plain string of JSON
SELECT * FROM raw_events;

## Concept 1: Dot Notation for STRUCT Columns

When a column has a **STRUCT** type (a column containing named sub-fields), you access nested fields using **dot notation** — just like accessing properties in JavaScript or Python.

```sql
SELECT address.city FROM customer_profiles;
```

| Syntax | What it means |
| --- | --- |
| **`address.city`** | Reach into the `address` struct and pull out the `city` field |
| **`address.state`** | Same struct, different field |
| **`preferences.loyalty_tier`** | A different struct column, accessing its `loyalty_tier` field |

This works because `address` is a **typed** column — Spark knows at query time exactly what fields are inside it. No parsing needed.

**Power BI analogy:** Think of this like expanding a nested table in Power Query. But instead of clicking UI buttons, you just use dot notation.

---

In [0]:
%sql
-- ============================================================
-- Dot notation: reaching into STRUCT columns
-- ============================================================
-- address.city, address.state, address.zipcode are all nested inside
-- the 'address' column. We access them with simple dot notation.

SELECT 
  name,
  address.street,
  address.city,
  address.state,
  address.zipcode,
  preferences.favorite_category,
  preferences.loyalty_tier
FROM customer_profiles;

In [0]:
%sql
-- You can filter on nested fields too — they work in WHERE clauses
SELECT name, address.city, preferences.loyalty_tier
FROM customer_profiles
WHERE preferences.loyalty_tier = 'Gold'
  AND address.state IN ('OR', 'TX');

## Concept 2: Colon Syntax for Raw JSON Strings

Sometimes your data arrives as a **raw JSON string** rather than a properly typed struct. This is common with event streams, API responses, or ingested log files.

For these cases, Databricks supports **colon syntax** — a shorthand for extracting fields directly from a JSON string column:

```sql
SELECT payload:product_name::STRING FROM raw_events;
```

| Part | What it means |
| --- | --- |
| **`payload:product_name`** | Reach into the JSON string and extract the `product_name` field |
| **`::STRING`** | Cast the result to a STRING type (JSON values are untyped by default) |
| **`::DOUBLE`** | Cast to a number |
| **`::INT`** | Cast to an integer |

**Key difference from dot notation:** Dot notation works on typed STRUCT columns. Colon syntax works on **plain STRING columns** that happen to contain JSON. Think of colon syntax as the quick-and-dirty approach for ad-hoc exploration.

---

In [0]:
%sql
-- ============================================================
-- Colon syntax: extracting fields from a raw JSON string
-- ============================================================
-- The 'payload' column is just a STRING containing JSON.
-- We use :field_name to reach into it, and ::TYPE to cast the result.

SELECT 
  event_id,
  event_type,
  payload:product_name::STRING AS product_name,
  payload:amount::DOUBLE AS amount,
  payload:currency::STRING AS currency,
  payload:quantity::INT AS quantity
FROM raw_events
WHERE event_type = 'purchase';

In [0]:
%sql
-- Colon syntax works for different event types too
-- Each event type has different JSON fields — that's fine!
SELECT 
  event_id,
  event_type,
  payload:page::STRING AS page,
  payload:referrer::STRING AS referrer,
  payload:duration_seconds::INT AS duration_seconds
FROM raw_events
WHERE event_type = 'page_view';

## Concept 3: from_json() — Parsing JSON into a Typed Structure

Colon syntax is great for quick exploration. But for production queries and better performance, you want to convert that raw JSON string into a **properly typed STRUCT**.

`from_json()` takes two arguments:
1. The string column containing JSON
2. A **DDL schema** describing the expected structure

```sql
from_json(payload, 'product_id INT, product_name STRING, amount DOUBLE, currency STRING, quantity INT')
```

Once parsed, the result is a STRUCT — and now you can use **dot notation** on it, get proper type checking, and benefit from query optimisation.

**Power BI analogy:** This is like using `Json.Document()` in Power Query to parse a JSON column, then expanding the resulting record into typed columns.

---

In [0]:
%sql
-- ============================================================
-- from_json: converting a raw JSON string into a typed STRUCT
-- ============================================================
-- We provide a DDL schema string describing the expected structure.
-- The result is a STRUCT that we can then access with dot notation.

SELECT 
  event_id,
  event_type,
  from_json(
    payload, 
    'product_id INT, product_name STRING, amount DOUBLE, currency STRING, quantity INT'
  ) AS parsed
FROM raw_events
WHERE event_type = 'purchase';

In [0]:
%sql
-- Once parsed, you can use dot notation on the result
-- and do proper typed calculations (like summing amounts)
SELECT 
  event_id,
  from_json(
    payload, 
    'product_id INT, product_name STRING, amount DOUBLE, currency STRING, quantity INT'
  ).product_name AS product_name,
  from_json(
    payload, 
    'product_id INT, product_name STRING, amount DOUBLE, currency STRING, quantity INT'
  ).amount * from_json(
    payload, 
    'product_id INT, product_name STRING, amount DOUBLE, currency STRING, quantity INT'
  ).quantity AS total_line_value
FROM raw_events
WHERE event_type = 'purchase';

## Concept 4: to_json() — Collapsing a Structure Back to a String

`to_json()` is the reverse of `from_json()`. It takes a STRUCT (or MAP, or ARRAY) column and converts it back into a JSON string.

When would you use this? When sending data to an external system that expects JSON, or when writing to a table that stores payloads as strings.

```sql
SELECT to_json(address) AS address_json FROM customer_profiles;
```

---

In [0]:
%sql
-- ============================================================
-- to_json: converting a STRUCT back into a JSON string
-- ============================================================
-- Useful for exporting data or writing to systems that expect JSON

SELECT 
  name,
  to_json(address) AS address_json,
  to_json(preferences) AS preferences_json
FROM customer_profiles;

## Concept 5: Arrays and EXPLODE

Some columns contain **arrays** (lists of values). The `customer_profiles` table has a `tags` column that holds multiple tags per customer.

To work with arrays, you have two main options:

| Function | What it does |
| --- | --- |
| **`array_contains(col, value)`** | Returns TRUE if the array contains a specific value (like a filter) |
| **`EXPLODE(col)`** | Creates one row per array element ("unpivots" the array) |
| **`SIZE(col)`** | Returns the number of elements in the array |

`EXPLODE` is especially powerful — it turns one row with an array of 3 items into 3 rows, each with one item. This is like “expand to new rows” in Power Query.

---

In [0]:
%sql
-- ============================================================
-- Arrays: filtering with array_contains
-- ============================================================
-- Find all VIP customers (customers whose tags array contains 'vip')

SELECT 
  name,
  preferences.loyalty_tier,
  tags
FROM customer_profiles
WHERE array_contains(tags, 'vip');

In [0]:
%sql
-- ============================================================
-- EXPLODE: creating one row per array element
-- ============================================================
-- This unpivots the tags array into individual rows.
-- One customer with 3 tags becomes 3 rows.

SELECT 
  name,
  preferences.loyalty_tier,
  EXPLODE(tags) AS tag
FROM customer_profiles;

In [0]:
%sql
-- After exploding, you can aggregate
-- Which tags are most common across our customers?
SELECT 
  tag,
  COUNT(*) AS customer_count
FROM (
  SELECT EXPLODE(tags) AS tag FROM customer_profiles
)
GROUP BY tag
ORDER BY customer_count DESC;

---

## Summary: When to Use What

| Technique | Use When | Example |
| --- | --- | --- |
| **Dot notation** (`col.field`) | Column is a typed STRUCT | `address.city` |
| **Colon syntax** (`col:field::TYPE`) | Column is a raw JSON string | `payload:amount::DOUBLE` |
| **`from_json(col, schema)`** | You want to parse a JSON string into a typed STRUCT for better performance | `from_json(payload, 'amount DOUBLE, name STRING')` |
| **`to_json(col)`** | You want to collapse a STRUCT/ARRAY back into a JSON string for export | `to_json(address)` |
| **`EXPLODE(col)`** | You want to turn an array into individual rows | `EXPLODE(tags) AS tag` |
| **`array_contains(col, val)`** | You want to filter rows based on array membership | `array_contains(tags, 'vip')` |

**The key takeaway:** In a traditional database, semi-structured data is painful. In Databricks SQL, it’s a first-class citizen. Whether your data arrives as properly typed structs or raw JSON blobs, you have clean, readable syntax to work with it.

**Power BI comparison:** In Power Query, you’d use `Json.Document()` to parse, then expand columns through UI clicks. In Databricks SQL, you use dot notation, colon syntax, or `from_json()` — all in a single SQL statement.